# Gemma 4 E4B Fine-tuning — Backend/AI Expert

**Model:** `unsloth/gemma-4-E4B-it` → `AditHash/gemma4-backend-ai-expert`  
**Dataset:** `AditHash/backend-ai-instruct`  
**Platform:** Google Colab T4 (free tier)  
**Method:** QLoRA 4-bit via Unsloth

> **Text-only fine-tuning.** Uses `FastModel`, not `FastVisionModel`.

## Step 1 — Install dependencies

In [ ]:
# Install Unsloth (Colab-optimized build)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install datasets huggingface-hub rouge-score groq -q

## Step 2 — Authenticate with HuggingFace

In [ ]:
import os
from google.colab import userdata

# Store HF_TOKEN in Colab Secrets (key icon in left sidebar)
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

from huggingface_hub import login
login(token=HF_TOKEN)

## Step 3 — Configuration

In [ ]:
# All hyperparameters — edit here
BASE_MODEL = "unsloth/gemma-4-E4B-it"
DATASET_REPO = "AditHash/backend-ai-instruct"
HUB_MODEL_ID = "AditHash/gemma4-backend-ai-expert"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True          # QLoRA — fits on T4 16GB

LORA_RANK = 16
LORA_ALPHA = 16
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]
LORA_DROPOUT = 0.0

EPOCHS = 3
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4    # Effective batch = 8
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.03
LR_SCHEDULER = "cosine"
OPTIMIZER = "adamw_8bit"

# CRITICAL: Always 'unsloth' — fixes KV cache bug + reduces VRAM
USE_GRADIENT_CHECKPOINTING = "unsloth"

# 'gemma-4' for E2B/E4B (non-thinking). 'gemma-4-thinking' only for 26B/31B.
CHAT_TEMPLATE = "gemma-4"

OUTPUT_DIR = "./outputs"
print("Config loaded.")

## Step 4 — Load model + apply LoRA

In [ ]:
from unsloth import FastModel  # Text-only — NOT FastVisionModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    # do NOT set use_cache=False — Unsloth handles KV cache bugs automatically
)

model = FastModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    random_state=42,
)

tokenizer = get_chat_template(tokenizer, chat_template=CHAT_TEMPLATE)
print(f"Model loaded. Trainable params: {model.num_parameters(only_trainable=True):,}")

## Step 5 — Load + format dataset

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset(DATASET_REPO, split="train")
print(f"Raw samples: {len(raw_dataset)}")

def format_sample(sample):
    messages = [
        {"role": "user", "content": sample["instruction"]},
        {"role": "assistant", "content": sample["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = raw_dataset.map(format_sample, remove_columns=raw_dataset.column_names)
print(f"Formatted samples: {len(dataset)}")
print("Sample preview:")
print(dataset[0]["text"][:500])

## Step 6 — Train

In [ ]:
import torch
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        lr_scheduler_type=LR_SCHEDULER,
        optim=OPTIMIZER,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy="epoch",
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)

# Normal loss for E4B: 13–15. If you see 100+, gradient accumulation bug.
# If 0.0, data pipeline issue. Both are fixed by using Unsloth.
trainer_stats = trainer.train()
print(f"\nTraining complete. Final loss: {trainer_stats.training_loss:.4f}")

## Step 7 — Test inference

In [ ]:
FastModel.for_inference(model)

test_question = "How do you implement rate limiting per user in FastAPI?"

messages = [{"role": "user", "content": test_question}]
inputs = tokenizer.apply_chat_template(
    messages, return_tensors="pt", add_generation_prompt=True
).to(model.device)

outputs = model.generate(inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print(f"Q: {test_question}\n\nA: {response}")

## Step 8 — Push adapter to HuggingFace Hub

In [ ]:
model.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
tokenizer.push_to_hub(HUB_MODEL_ID, token=HF_TOKEN)
print(f"Pushed to https://huggingface.co/{HUB_MODEL_ID}")